# Earnings Announcement Tick Visualization

This notebook visualizes how option prices fluctuate around earnings announcements using tick data.

**Workflow:**
1. Input a symbol and determine its earnings date/timing
2. Identify the calendar spread strategy (strikes, expirations)
3. Load tick data around entry and exit windows
4. Plot bid/ask spreads and midpoint evolution for both legs
5. Visualize price impact of earnings announcement

**Prerequisites:**
- Earnings calendar data loaded
- Tick data backfilled for the symbol
- Option chain snapshots available

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, date, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from dlt_ibapi.repositories import (
    EarningsCalendarReader,
    OptionChainSnapshotReader,
    OptionTicksReader,
    EquityBarsReader,
)
from dlt_ibapi.strategies import EarningsTimingCalculator

# Rich for better output
from rich import print as rprint
from rich.console import Console

console = Console()

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Set pandas display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. Configuration

Specify the symbol to analyze.

In [2]:
# Configuration
SYMBOL = 'ARMK'  # Symbol to analyze

# Data paths
data_dir = project_root / "data_delta"

rprint(f"[bold cyan]Earnings Tick Visualization:[/bold cyan]")
rprint(f"  Symbol: {SYMBOL}")
rprint(f"  Data Directory: {data_dir}")

Earnings Tick Visualization:

Symbol: ARMK

Data Directory: /Users/mohamedali/trading_project/dlt-ibapi/data_delta

## 2. Load Earnings Information

Find the earnings date and timing for the symbol.

In [3]:
# Load earnings calendar
earnings_reader = EarningsCalendarReader(str(data_dir), "earnings")

# Get earnings for symbol
earnings_df = earnings_reader.get_earnings_for_symbol(
    symbol=SYMBOL,
    start_date=date(2025, 11, 1),
    end_date=date(2025, 12, 31)
)

if earnings_df.empty:
    rprint(f"[bold red]No earnings found for {SYMBOL}[/bold red]")
    raise ValueError(f"No earnings data for {SYMBOL}")

# Use the first (most recent) earnings
earnings_date = earnings_df.iloc[0]['earnings_date']
earnings_timing = earnings_df.iloc[0]['earnings_time']
company_name = earnings_df.iloc[0]['company_name']

rprint(f"\n[bold green]Earnings Found:[/bold green]")
rprint(f"  Company: {company_name}")
rprint(f"  Date: {earnings_date}")
rprint(f"  Timing: {earnings_timing}")

# Calculate entry/exit windows
calculator = EarningsTimingCalculator()
windows = calculator.calculate_windows(
    earnings_date=earnings_date,
    earnings_time=earnings_timing,
)

rprint(f"\n[bold cyan]Trading Windows:[/bold cyan]")
rprint(f"  Entry: {windows.entry.start} to {windows.entry.end}")
rprint(f"  Exit: {windows.exit.start} to {windows.exit.end}")

Earnings Found:

Company: Aramark

Date: 2025-11-17

Timing: PRE_MARKET

Trading Windows:

Entry: 2025-11-14 15:00:00 to 2025-11-14 16:00:00

Exit: 2025-11-17 09:00:00 to 2025-11-17 10:00:00

## 3. Determine Calendar Spread Strategy

Find the strikes and expirations for the calendar spread.

In [4]:
# Get option chain snapshot
chain_reader = OptionChainSnapshotReader(str(data_dir), "option_chains")

# Get available expirations
try:
    expirations = chain_reader.get_available_expirations(
        underlying=SYMBOL,
        as_of=earnings_date,
    )
    
    if len(expirations) < 2:
        rprint(f"[bold red]Need at least 2 expirations for calendar spread[/bold red]")
        raise ValueError("Insufficient expirations")
    
    # Use first two expirations (short and long)
    short_expiry = expirations[0]
    long_expiry = expirations[1]
    
    rprint(f"\n[bold cyan]Expirations:[/bold cyan]")
    rprint(f"  Short: {short_expiry}")
    rprint(f"  Long: {long_expiry}")
    
except Exception as e:
    rprint(f"[bold red]Error loading option chain: {e}[/bold red]")
    # Fallback to manual specification
    short_expiry = date(2025, 11, 21)
    long_expiry = date(2025, 12, 19)
    rprint(f"[bold yellow]Using fallback expirations: {short_expiry}, {long_expiry}[/bold yellow]")

# Get spot price to determine strike
equity_reader = EquityBarsReader(str(data_dir), "stocks")
try:
    spot_bars = equity_reader.get_bars(
        symbol=SYMBOL,
        bar_size="1 day",
        start_date=windows.spot_price_date,
        end_date=windows.spot_price_date,
    )
    
    if not spot_bars.empty:
        spot_price = spot_bars.iloc[0]['close']
        rprint(f"\n[bold cyan]Spot Price:[/bold cyan]")
        rprint(f"  Date: {windows.spot_price_date}")
        rprint(f"  Price: ${spot_price:.2f}")
    else:
        spot_price = None
        rprint(f"\n[bold yellow]No spot price data found[/bold yellow]")
except Exception as e:
    spot_price = None
    rprint(f"\n[bold yellow]Error loading spot price: {e}[/bold yellow]")

# Try to get ATM strike from option chain snapshot
strike = None
right = 'C'  # Call option

if spot_price is not None:
    # Round DOWN to nearest $5 (not standard rounding - use floor division)
    strike = int(spot_price // 5) * 5
    if spot_price % 5 > 2.5:  # If closer to next strike, round up
        strike += 5
    rprint(f"\n[bold green]Strike from spot price: ${strike:.1f}[/bold green]")
else:
    # Try to get ATM strike from option chain
    try:
        strikes = chain_reader.get_strikes_for_expiry(
            underlying=SYMBOL,
            as_of=earnings_date,
            expiry=short_expiry
        )
        if strikes:
            # Get middle strike (closest to ATM)
            strike = strikes[len(strikes) // 2]
            rprint(f"\n[bold green]Strike from option chain (middle): ${strike:.1f}[/bold green]")
    except:
        pass

# Fallback to manual specification
if strike is None:
    strike = 38.0  # Manual fallback for ARMK
    rprint(f"\n[bold yellow]Using fallback strike: ${strike:.1f}[/bold yellow]")

strike = 38.0

rprint(f"\n[bold green]Calendar Spread Strategy:[/bold green]")
rprint(f"  Strike: ${strike:.1f}")
rprint(f"  Right: {right}")
rprint(f"  Short: {SYMBOL} ${strike}{right} exp {short_expiry}")
rprint(f"  Long: {SYMBOL} ${strike}{right} exp {long_expiry}")

Expirations:

Short: 2025-11-21

Long: 2025-12-19

Error loading spot price: 'EarningsTradeWindows' object has no attribute 'spot_price_date'

Strike from option chain (middle): $40.0

Calendar Spread Strategy:

Strike: $38.0

Right: C

Short: ARMK $38.0C exp 2025-11-21

Long: ARMK $38.0C exp 2025-12-19

## 4. Load Tick Data

Load bid/ask tick data for both legs around entry and exit windows.

In [5]:
# Initialize tick reader
# Note: Dataset name is "option_ticks" not "ticks"
tick_reader = OptionTicksReader(str(data_dir), "option_ticks")

# Load tick data for short leg
rprint(f"\n[bold cyan]Loading tick data for short leg...[/bold cyan]")
short_entry_ticks = tick_reader.get_ticks(
    underlying=SYMBOL,
    expiry=short_expiry,
    strike=strike,
    right=right,
    start_datetime=windows.entry.start,
    end_datetime=windows.entry.end,
    tick_type='bid_ask'
)

short_exit_ticks = tick_reader.get_ticks(
    underlying=SYMBOL,
    expiry=short_expiry,
    strike=strike,
    right=right,
    start_datetime=windows.exit.start,
    end_datetime=windows.exit.end,
    tick_type='bid_ask'
)

rprint(f"  Entry: {len(short_entry_ticks)} ticks")
rprint(f"  Exit: {len(short_exit_ticks)} ticks")

# Load tick data for long leg
rprint(f"\n[bold cyan]Loading tick data for long leg...[/bold cyan]")
long_entry_ticks = tick_reader.get_ticks(
    underlying=SYMBOL,
    expiry=long_expiry,
    strike=strike,
    right=right,
    start_datetime=windows.entry.start,
    end_datetime=windows.entry.end,
    tick_type='bid_ask'
)

long_exit_ticks = tick_reader.get_ticks(
    underlying=SYMBOL,
    expiry=long_expiry,
    strike=strike,
    right=right,
    start_datetime=windows.exit.start,
    end_datetime=windows.exit.end,
    tick_type='bid_ask'
)

rprint(f"  Entry: {len(long_entry_ticks)} ticks")
rprint(f"  Exit: {len(long_exit_ticks)} ticks")

# Combine entry and exit for each leg
short_leg_ticks = pd.concat([short_entry_ticks, short_exit_ticks], ignore_index=True).sort_values('tick_time')
long_leg_ticks = pd.concat([long_entry_ticks, long_exit_ticks], ignore_index=True).sort_values('tick_time')

rprint(f"\n[bold green]Total ticks loaded:[/bold green]")
rprint(f"  Short leg: {len(short_leg_ticks)} ticks")
rprint(f"  Long leg: {len(long_leg_ticks)} ticks")

Loading tick data for short leg...

Entry: 0 ticks

Exit: 0 ticks

Loading tick data for long leg...

Entry: 0 ticks

Exit: 0 ticks

Total ticks loaded:

Short leg: 0 ticks

Long leg: 0 ticks

## 5. Plot Short Leg (Near-Term Expiration)

Visualize bid/ask spread and midpoint evolution for the short leg.

In [6]:
def plot_option_ticks(ticks_df, leg_name, expiry, windows, earnings_date, earnings_timing):
    """
    Plot bid/ask tick data with entry/exit windows highlighted.
    
    Args:
        ticks_df: DataFrame with tick data
        leg_name: 'Short' or 'Long'
        expiry: Expiration date
        windows: EarningsTimingWindows object
        earnings_date: Earnings announcement date
        earnings_timing: 'PRE_MARKET' or 'AFTER_HOURS'
    """
    if ticks_df.empty:
        rprint(f"[bold red]No tick data to plot for {leg_name} leg[/bold red]")
        return
    
    # Calculate midpoint
    ticks_df['midpoint'] = (ticks_df['bid_price'] + ticks_df['ask_price']) / 2
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
    
    # Convert tick_time to datetime if needed
    if not pd.api.types.is_datetime64_any_dtype(ticks_df['tick_time']):
        ticks_df['tick_time'] = pd.to_datetime(ticks_df['tick_time'])
    
    # Plot 1: Bid/Ask/Midpoint
    ax1.plot(ticks_df['tick_time'], ticks_df['bid_price'], 'b-', alpha=0.6, linewidth=1, label='Bid')
    ax1.plot(ticks_df['tick_time'], ticks_df['ask_price'], 'r-', alpha=0.6, linewidth=1, label='Ask')
    ax1.plot(ticks_df['tick_time'], ticks_df['midpoint'], 'g-', linewidth=2, label='Midpoint')
    ax1.fill_between(ticks_df['tick_time'], ticks_df['bid_price'], ticks_df['ask_price'], alpha=0.2, color='gray')
    
    # Highlight entry window
    entry_start = pd.to_datetime(windows.entry.start)
    entry_end = pd.to_datetime(windows.entry.end)
    ax1.axvspan(entry_start, entry_end, alpha=0.2, color='blue', label='Entry Window')
    
    # Highlight exit window
    exit_start = pd.to_datetime(windows.exit.start)
    exit_end = pd.to_datetime(windows.exit.end)
    ax1.axvspan(exit_start, exit_end, alpha=0.2, color='orange', label='Exit Window')
    
    # Mark earnings announcement time
    if earnings_timing == 'PRE_MARKET':
        earnings_time = pd.Timestamp(earnings_date) + pd.Timedelta(hours=8)  # 8am
        label = f'Earnings (Pre-Market {earnings_date})'
    else:
        earnings_time = pd.Timestamp(earnings_date) + pd.Timedelta(hours=16)  # 4pm
        label = f'Earnings (After-Hours {earnings_date})'
    
    ax1.axvline(earnings_time, color='red', linestyle='--', linewidth=2, label=label)
    
    ax1.set_ylabel('Price ($)', fontsize=12, fontweight='bold')
    ax1.set_title(f'{leg_name} Leg - Bid/Ask Spread (Exp: {expiry})', fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Spread percentage
    ax2.plot(ticks_df['tick_time'], ticks_df['spread_pct'], 'purple', linewidth=1.5, label='Spread %')
    ax2.axvspan(entry_start, entry_end, alpha=0.2, color='blue')
    ax2.axvspan(exit_start, exit_end, alpha=0.2, color='orange')
    ax2.axvline(earnings_time, color='red', linestyle='--', linewidth=2)
    
    ax2.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Spread %', fontsize=12, fontweight='bold')
    ax2.set_title(f'{leg_name} Leg - Bid/Ask Spread Percentage', fontsize=14, fontweight='bold')
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    # Format x-axis
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    rprint(f"\n[bold cyan]{leg_name} Leg Statistics:[/bold cyan]")
    rprint(f"  Avg Bid: ${ticks_df['bid_price'].mean():.4f}")
    rprint(f"  Avg Ask: ${ticks_df['ask_price'].mean():.4f}")
    rprint(f"  Avg Midpoint: ${ticks_df['midpoint'].mean():.4f}")
    rprint(f"  Avg Spread: ${ticks_df['spread'].mean():.4f}")
    rprint(f"  Avg Spread %: {ticks_df['spread_pct'].mean():.2f}%")
    rprint(f"  Price Range: ${ticks_df['midpoint'].min():.4f} - ${ticks_df['midpoint'].max():.4f}")
    
    # Entry vs Exit comparison
    entry_ticks = ticks_df[(ticks_df['tick_time'] >= entry_start) & (ticks_df['tick_time'] <= entry_end)]
    exit_ticks = ticks_df[(ticks_df['tick_time'] >= exit_start) & (ticks_df['tick_time'] <= exit_end)]
    
    if not entry_ticks.empty and not exit_ticks.empty:
        entry_mid = entry_ticks['midpoint'].mean()
        exit_mid = exit_ticks['midpoint'].mean()
        change = exit_mid - entry_mid
        change_pct = (change / entry_mid) * 100
        
        rprint(f"\n[bold green]Entry → Exit Change:[/bold green]")
        rprint(f"  Entry Avg: ${entry_mid:.4f}")
        rprint(f"  Exit Avg: ${exit_mid:.4f}")
        rprint(f"  Change: ${change:.4f} ({change_pct:+.2f}%)")

# Plot short leg
plot_option_ticks(
    ticks_df=short_leg_ticks,
    leg_name='Short',
    expiry=short_expiry,
    windows=windows,
    earnings_date=earnings_date,
    earnings_timing=earnings_timing
)

No tick data to plot for Short leg

## 6. Plot Long Leg (Longer-Term Expiration)

Visualize bid/ask spread and midpoint evolution for the long leg.

In [7]:
# Plot long leg
plot_option_ticks(
    ticks_df=long_leg_ticks,
    leg_name='Long',
    expiry=long_expiry,
    windows=windows,
    earnings_date=earnings_date,
    earnings_timing=earnings_timing
)

No tick data to plot for Long leg

## 7. Calendar Spread Net Value

Plot the net value of the calendar spread (Long - Short) over time.

In [8]:
if not short_leg_ticks.empty and not long_leg_ticks.empty:
    # Merge on tick_time to align data
    # Use asof merge to handle non-exact timestamp matches
    short_leg_ticks = short_leg_ticks.sort_values('tick_time').reset_index(drop=True)
    long_leg_ticks = long_leg_ticks.sort_values('tick_time').reset_index(drop=True)
    
    # Calculate midpoint for both
    short_leg_ticks['midpoint'] = (short_leg_ticks['bid_price'] + short_leg_ticks['ask_price']) / 2
    long_leg_ticks['midpoint'] = (long_leg_ticks['bid_price'] + long_leg_ticks['ask_price']) / 2
    
    # Merge using asof (forward fill)
    merged = pd.merge_asof(
        long_leg_ticks[['tick_time', 'midpoint']].rename(columns={'midpoint': 'long_mid'}),
        short_leg_ticks[['tick_time', 'midpoint']].rename(columns={'midpoint': 'short_mid'}),
        on='tick_time',
        direction='nearest',
        tolerance=pd.Timedelta('30s')
    )
    
    # Calculate spread net value (Long - Short)
    merged['spread_value'] = merged['long_mid'] - merged['short_mid']
    merged = merged.dropna()
    
    if not merged.empty:
        # Plot
        fig, ax = plt.subplots(figsize=(16, 6))
        
        ax.plot(merged['tick_time'], merged['spread_value'], 'darkgreen', linewidth=2, label='Calendar Spread Value (Long - Short)')
        ax.fill_between(merged['tick_time'], 0, merged['spread_value'], alpha=0.3, color='green')
        
        # Highlight windows
        entry_start = pd.to_datetime(windows.entry.start)
        entry_end = pd.to_datetime(windows.entry.end)
        exit_start = pd.to_datetime(windows.exit.start)
        exit_end = pd.to_datetime(windows.exit.end)
        
        ax.axvspan(entry_start, entry_end, alpha=0.2, color='blue', label='Entry Window')
        ax.axvspan(exit_start, exit_end, alpha=0.2, color='orange', label='Exit Window')
        
        # Mark earnings
        if earnings_timing == 'PRE_MARKET':
            earnings_time = pd.Timestamp(earnings_date) + pd.Timedelta(hours=8)
            label = f'Earnings (Pre-Market {earnings_date})'
        else:
            earnings_time = pd.Timestamp(earnings_date) + pd.Timedelta(hours=16)
            label = f'Earnings (After-Hours {earnings_date})'
        
        ax.axvline(earnings_time, color='red', linestyle='--', linewidth=2, label=label)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
        
        ax.set_xlabel('Time', fontsize=12, fontweight='bold')
        ax.set_ylabel('Spread Value ($)', fontsize=12, fontweight='bold')
        ax.set_title(f'{SYMBOL} Calendar Spread Net Value (${strike}{right})', fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        
        # Format x-axis
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
        plt.xticks(rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()
        
        # Statistics
        entry_spread = merged[(merged['tick_time'] >= entry_start) & (merged['tick_time'] <= entry_end)]['spread_value'].mean()
        exit_spread = merged[(merged['tick_time'] >= exit_start) & (merged['tick_time'] <= exit_end)]['spread_value'].mean()
        
        rprint(f"\n[bold cyan]Calendar Spread Statistics:[/bold cyan]")
        rprint(f"  Entry Avg Spread: ${entry_spread:.4f}")
        rprint(f"  Exit Avg Spread: ${exit_spread:.4f}")
        rprint(f"  Change: ${exit_spread - entry_spread:.4f}")
        rprint(f"  P&L per contract: ${(exit_spread - entry_spread) * 100:.2f}")
    else:
        rprint("[bold red]No overlapping tick data for calendar spread calculation[/bold red]")
else:
    rprint("[bold red]Insufficient tick data to plot calendar spread[/bold red]")

Insufficient tick data to plot calendar spread

## 8. Key Insights

Summary of price movements and observations.

In [9]:
rprint("\n[bold cyan]KEY INSIGHTS:[/bold cyan]")
rprint("\n[bold green]Short Leg (Near-Term):[/bold green]")
rprint("  • More sensitive to earnings announcement")
rprint("  • Higher IV crush expected after announcement")
rprint("  • Wider spreads indicate lower liquidity")

rprint("\n[bold green]Long Leg (Longer-Term):[/bold green]")
rprint("  • Less sensitive to immediate earnings impact")
rprint("  • Retains more time value")
rprint("  • Generally tighter spreads (higher liquidity)")

rprint("\n[bold green]Calendar Spread Strategy:[/bold green]")
rprint("  • Profits from IV differential between expirations")
rprint("  • Entry: Sell short-term (high IV) + Buy long-term (lower IV)")
rprint("  • Exit: After earnings announcement when IV normalizes")
rprint("  • Success depends on short leg IV dropping more than long leg")

KEY INSIGHTS:

Short Leg (Near-Term):

• More sensitive to earnings announcement

• Higher IV crush expected after announcement

• Wider spreads indicate lower liquidity

...

Calendar Spread Strategy:

• Profits from IV differential between expirations

• Entry: Sell short-term (high IV) + Buy long-term (lower IV)

• Exit: After earnings announcement when IV normalizes

• Success depends on short leg IV dropping more than long leg